# HMM Categorical V1 — NVIDIA Developer Journey

**Goal:** Build the first categorical HMM using the weekly GMM cluster table.

This notebook uses:

- `dev_gmm_weekly_clusters_v1` as the weekly observed behavior sequence.
- `dev_lifecycle_cluster_membership_v11_final` as the static lifecycle/HDBSCAN interpretation layer.

Conceptually:

```text
Weekly GMM cluster = observed weekly behavior state
Categorical HMM = hidden journey-state transition model
V11 lifecycle/HDBSCAN cluster = static developer segment used for business interpretation
```

This notebook is designed to run locally or in Google Colab Pro/Pro+.

In [ ]:
# ============================================================
# Optional Colab setup
# ============================================================

# If running in Colab, uncomment these lines:
# from google.colab import drive
# drive.mount('/content/drive')

# Then set PROJECT_DIR to wherever your DuckDB/parquet files live in Drive.
# Example:
# PROJECT_DIR = "/content/drive/MyDrive/Spring2026_IndustryProject"

# If running locally, keep PROJECT_DIR as "."
PROJECT_DIR = "."

print("PROJECT_DIR:", PROJECT_DIR)

In [ ]:
# ============================================================
# Install packages if needed
# ============================================================

# In Colab, upload requirements_hmm_categorical_v1.txt or place it in PROJECT_DIR,
# then uncomment one of the following:

# !pip install -r requirements_hmm_categorical_v1.txt

# Or install directly:
# !pip install duckdb pandas numpy scikit-learn hmmlearn matplotlib pyarrow

In [ ]:
# ============================================================
# Imports
# ============================================================

import os
from pathlib import Path
import warnings

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from hmmlearn.hmm import CategoricalHMM

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

print("Imports loaded.")

In [ ]:
# ============================================================
# Paths and configurable parameters
# ============================================================

PROJECT_DIR = Path(PROJECT_DIR)

DB_PATH = PROJECT_DIR / "developer_project.duckdb"

# Folder containing exported parquet files, if you are loading tables from parquet.
# Update this path if needed.
PARQUET_DIR = PROJECT_DIR / "outputs" / "v11_cluster_exports"

GMM_WEEKLY_TABLE = "dev_gmm_weekly_clusters_v1"
V11_FINAL_TABLE = "dev_lifecycle_cluster_membership_v11_final"

# First-pass modeling choices
VALID_STRATA = ["active", "cooling", "at_risk"]  # use ["active", "at_risk"] if cooling is empty
MIN_WEEKS_PER_DEV = 6
MAX_DEVELOPERS = 25000
MIN_GMM_POSTERIOR = 0.50

# HMM candidates
N_HIDDEN_STATE_OPTIONS = [2, 3, 4, 5]

# Output directory for small experiment summaries
OUTPUT_DIR = PROJECT_DIR / "outputs" / "hmm_categorical_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DB_PATH:", DB_PATH)
print("PARQUET_DIR:", PARQUET_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# ============================================================
# Connect to DuckDB
# ============================================================

con = duckdb.connect(str(DB_PATH))
print("Connected to DuckDB:", DB_PATH)

## Load required tables from Parquet if needed

This cell is useful in Colab if your DuckDB file does not already contain the tables. It safely uses `CREATE OR REPLACE TABLE`.

Expected parquet filenames:

```text
dev_gmm_weekly_clusters_v1.parquet
dev_lifecycle_cluster_membership_v11_final.parquet
```

If your DuckDB already has these tables, this cell will simply skip loading unless you set `LOAD_FROM_PARQUET = True`.

In [ ]:
# ============================================================
# Optional: Load required tables from parquet
# ============================================================

LOAD_FROM_PARQUET = False  # Change to True if running in Colab from parquet exports

required_parquets = {
    GMM_WEEKLY_TABLE: PARQUET_DIR / f"{GMM_WEEKLY_TABLE}.parquet",
    V11_FINAL_TABLE: PARQUET_DIR / f"{V11_FINAL_TABLE}.parquet",
}

if LOAD_FROM_PARQUET:
    for table_name, parquet_path in required_parquets.items():
        if parquet_path.exists():
            print(f"Loading {parquet_path.name} into {table_name}...")
            con.execute(f"""
                CREATE OR REPLACE TABLE {table_name} AS
                SELECT *
                FROM read_parquet('{parquet_path.as_posix()}')
            """)
            print(f"Loaded {table_name}.")
        else:
            raise FileNotFoundError(f"Missing parquet file: {parquet_path}")
else:
    print("LOAD_FROM_PARQUET=False. Using tables already present in DuckDB.")

In [ ]:
# ============================================================
# Validate required tables exist
# ============================================================

tables = con.execute("SHOW TABLES").df()
display(tables)

existing_tables = set(tables["name"].tolist())

for table in [GMM_WEEKLY_TABLE, V11_FINAL_TABLE]:
    if table not in existing_tables:
        raise ValueError(
            f"Required table not found: {table}. "
            "Either load it from parquet by setting LOAD_FROM_PARQUET=True, "
            "or make sure it already exists in developer_project.duckdb."
        )

print("Required tables found.")

In [ ]:
# ============================================================
# Inspect weekly GMM table
# ============================================================

gmm_summary = con.execute(f"""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT developer_id) AS n_developers,
    MIN(week_start) AS min_week,
    MAX(week_start) AS max_week,
    COUNT(DISTINCT gmm_weekly_cluster_id) AS n_gmm_clusters
FROM {GMM_WEEKLY_TABLE}
""").df()

display(gmm_summary)

gmm_dist = con.execute(f"""
SELECT
    gmm_weekly_cluster_id,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT developer_id) AS n_developers,
    AVG(gmm_weekly_max_posterior) AS avg_max_posterior
FROM {GMM_WEEKLY_TABLE}
GROUP BY gmm_weekly_cluster_id
ORDER BY gmm_weekly_cluster_id
""").df()

display(gmm_dist)

In [ ]:
# ============================================================
# Inspect V11 lifecycle/HDBSCAN table
# ============================================================

v11_summary = con.execute(f"""
SELECT
    stratum,
    COUNT(*) AS n_developers,
    COUNT(DISTINCT cluster_key) AS n_cluster_keys,
    AVG(cluster_probability) AS avg_cluster_probability,
    AVG(outlier_score) AS avg_outlier_score
FROM {V11_FINAL_TABLE}
GROUP BY stratum
ORDER BY n_developers DESC
""").df()

display(v11_summary)

In [ ]:
# ============================================================
# Check weekly GMM coverage by lifecycle stratum
# ============================================================

coverage_by_stratum = con.execute(f"""
SELECT
    c.stratum,
    COUNT(*) AS n_weekly_rows,
    COUNT(DISTINCT g.developer_id) AS n_developers,
    COUNT(DISTINCT g.gmm_weekly_cluster_id) AS n_gmm_clusters,
    AVG(g.gmm_weekly_max_posterior) AS avg_gmm_posterior
FROM {GMM_WEEKLY_TABLE} g
LEFT JOIN {V11_FINAL_TABLE} c
    ON g.developer_id = c.developer_id
GROUP BY c.stratum
ORDER BY n_developers DESC
""").df()

display(coverage_by_stratum)

In [ ]:
# ============================================================
# Create HMM input temp table
# ============================================================

valid_strata_sql = ", ".join([f"'{s}'" for s in VALID_STRATA])

con.execute(f"""
CREATE OR REPLACE TEMP TABLE hmm_gmm_input_temp AS
SELECT
    g.developer_id,
    CAST(g.week_start AS DATE) AS week_start,
    CAST(g.gmm_weekly_cluster_id AS INTEGER) AS gmm_weekly_cluster_id,
    CAST(g.gmm_weekly_max_posterior AS DOUBLE) AS gmm_weekly_max_posterior,
    c.stratum,
    c.cluster_key,
    c.cluster_probability,
    c.outlier_score,
    c.adoption_direction
FROM {GMM_WEEKLY_TABLE} g
JOIN {V11_FINAL_TABLE} c
    ON g.developer_id = c.developer_id
WHERE c.stratum IN ({valid_strata_sql})
  AND g.gmm_weekly_max_posterior >= {MIN_GMM_POSTERIOR}
""")

input_summary = con.execute("""
SELECT
    stratum,
    COUNT(*) AS n_weekly_rows,
    COUNT(DISTINCT developer_id) AS n_developers,
    COUNT(DISTINCT gmm_weekly_cluster_id) AS n_gmm_clusters,
    AVG(gmm_weekly_max_posterior) AS avg_gmm_posterior
FROM hmm_gmm_input_temp
GROUP BY stratum
ORDER BY n_developers DESC
""").df()

display(input_summary)

In [ ]:
# ============================================================
# Select developers with enough weekly observations
# ============================================================

eligible_devs = con.execute(f"""
WITH dev_counts AS (
    SELECT
        developer_id,
        COUNT(*) AS n_weeks,
        AVG(gmm_weekly_max_posterior) AS avg_gmm_confidence
    FROM hmm_gmm_input_temp
    GROUP BY developer_id
    HAVING COUNT(*) >= {MIN_WEEKS_PER_DEV}
)
SELECT developer_id, n_weeks, avg_gmm_confidence
FROM dev_counts
ORDER BY random()
LIMIT {MAX_DEVELOPERS}
""").df()

print("Eligible sampled developers:", eligible_devs.shape[0])
display(eligible_devs.describe(include="all"))

con.register("eligible_devs_sample", eligible_devs)

In [ ]:
# ============================================================
# Load sampled sequence data into pandas
# ============================================================

hmm_df = con.execute("""
SELECT h.*
FROM hmm_gmm_input_temp h
JOIN eligible_devs_sample e
    ON h.developer_id = e.developer_id
ORDER BY h.developer_id, h.week_start
""").df()

print("hmm_df shape:", hmm_df.shape)
display(hmm_df.head())

In [ ]:
# ============================================================
# Remap GMM cluster IDs to contiguous integer observations
# CategoricalHMM requires observations to be integer categories.
# ============================================================

unique_clusters = sorted(hmm_df["gmm_weekly_cluster_id"].dropna().unique().tolist())

cluster_id_map = {old_id: new_id for new_id, old_id in enumerate(unique_clusters)}
reverse_cluster_id_map = {new_id: old_id for old_id, new_id in cluster_id_map.items()}

hmm_df["gmm_obs_id"] = hmm_df["gmm_weekly_cluster_id"].map(cluster_id_map).astype(int)

print("Original GMM cluster IDs:", unique_clusters)
print("Observation ID map:", cluster_id_map)
print("Number of observed GMM categories:", hmm_df["gmm_obs_id"].nunique())

display(
    hmm_df.groupby(["gmm_weekly_cluster_id", "gmm_obs_id"])
    .size()
    .reset_index(name="n_rows")
    .sort_values("gmm_obs_id")
)

In [ ]:
# ============================================================
# Build HMM X array and sequence lengths
# ============================================================

hmm_df = hmm_df.sort_values(["developer_id", "week_start"]).reset_index(drop=True)

lengths = hmm_df.groupby("developer_id").size().tolist()
X = hmm_df[["gmm_obs_id"]].astype(int).values

n_observed_gmm_clusters = hmm_df["gmm_obs_id"].nunique()

print("X shape:", X.shape)
print("Number of developer sequences:", len(lengths))
print("Total sequence length:", sum(lengths))
print("Average sequence length:", np.mean(lengths))
print("Median sequence length:", np.median(lengths))
print("Observed GMM categories:", n_observed_gmm_clusters)

assert sum(lengths) == X.shape[0]

In [ ]:
# ============================================================
# Fit Categorical HMM models
# ============================================================

results = []
models = {}

for n_states in N_HIDDEN_STATE_OPTIONS:
    print(f"\nFitting CategoricalHMM with {n_states} hidden states...")

    model = CategoricalHMM(
        n_components=n_states,
        n_features=n_observed_gmm_clusters,
        n_iter=100,
        tol=1e-4,
        random_state=42,
        verbose=False
    )

    model.fit(X, lengths)
    log_likelihood = model.score(X, lengths)

    # Approximate parameter count for AIC/BIC:
    # start probs: n_states - 1
    # transition rows: n_states * (n_states - 1)
    # emission rows: n_states * (n_observed_categories - 1)
    n_params = (
        (n_states - 1)
        + n_states * (n_states - 1)
        + n_states * (n_observed_gmm_clusters - 1)
    )

    n_obs = X.shape[0]
    aic = -2 * log_likelihood + 2 * n_params
    bic = -2 * log_likelihood + n_params * np.log(n_obs)

    results.append({
        "n_hidden_states": n_states,
        "log_likelihood": log_likelihood,
        "aic": aic,
        "bic": bic,
        "n_obs": n_obs,
        "n_sequences": len(lengths),
        "n_observed_gmm_clusters": n_observed_gmm_clusters,
        "converged": model.monitor_.converged,
        "n_iter": model.monitor_.iter
    })

    models[n_states] = model

results_df = pd.DataFrame(results).sort_values("bic")
display(results_df)

In [ ]:
# ============================================================
# Choose best model
# ============================================================

# Default: choose lowest BIC.
# You can override this manually after looking at interpretability.
BEST_N_STATES = int(results_df.iloc[0]["n_hidden_states"])

# Optional override:
# BEST_N_STATES = 4

best_model = models[BEST_N_STATES]

print("Selected hidden states:", BEST_N_STATES)

In [ ]:
# ============================================================
# Predict hidden journey states
# ============================================================

hmm_df["hmm_state"] = best_model.predict(X, lengths)

display(
    hmm_df[[
        "developer_id",
        "week_start",
        "stratum",
        "cluster_key",
        "gmm_weekly_cluster_id",
        "gmm_obs_id",
        "hmm_state"
    ]].head(20)
)

## Interpret hidden states

For a categorical HMM, the emission matrix is one of the most important interpretation tools.

It answers:

```text
When the model is in hidden HMM state X, what weekly GMM cluster is it likely to emit?
```

Use this to give business-friendly names to HMM states after looking at what GMM clusters 0/1/2 mean.

In [ ]:
# ============================================================
# Emission probabilities
# ============================================================

emission_probs = pd.DataFrame(
    best_model.emissionprob_,
    index=[f"hmm_state_{i}" for i in range(BEST_N_STATES)],
    columns=[f"gmm_obs_{i}_orig_{reverse_cluster_id_map[i]}" for i in range(n_observed_gmm_clusters)]
)

display(emission_probs)

# Most likely GMM emission for each HMM state
dominant_emissions = (
    emission_probs
    .idxmax(axis=1)
    .reset_index()
    .rename(columns={"index": "hmm_state", 0: "dominant_emission"})
)

display(dominant_emissions)

In [ ]:
# ============================================================
# Transition matrix
# ============================================================

transition_matrix = pd.DataFrame(
    best_model.transmat_,
    index=[f"from_hmm_state_{i}" for i in range(BEST_N_STATES)],
    columns=[f"to_hmm_state_{i}" for i in range(BEST_N_STATES)]
)

display(transition_matrix)

transition_long = (
    transition_matrix
    .reset_index()
    .melt(id_vars="index", var_name="to_state", value_name="transition_probability")
    .rename(columns={"index": "from_state"})
)

display(transition_long.sort_values("transition_probability", ascending=False).head(20))

In [ ]:
# ============================================================
# State profiles
# ============================================================

state_profiles = (
    hmm_df
    .groupby("hmm_state")
    .agg(
        n_weekly_rows=("developer_id", "size"),
        n_developers=("developer_id", "nunique"),
        avg_gmm_posterior=("gmm_weekly_max_posterior", "mean"),
        most_common_gmm_cluster=("gmm_weekly_cluster_id", lambda x: x.value_counts().index[0]),
        stratum_mode=("stratum", lambda x: x.value_counts().index[0])
    )
    .reset_index()
)

state_profiles["share_of_rows"] = state_profiles["n_weekly_rows"] / state_profiles["n_weekly_rows"].sum()

display(state_profiles)

In [ ]:
# ============================================================
# HMM state distribution by lifecycle stratum
# ============================================================

state_by_stratum = (
    hmm_df
    .groupby(["stratum", "hmm_state"])
    .size()
    .reset_index(name="n_rows")
)

state_by_stratum["share_within_stratum"] = (
    state_by_stratum["n_rows"]
    / state_by_stratum.groupby("stratum")["n_rows"].transform("sum")
)

display(state_by_stratum.sort_values(["stratum", "hmm_state"]))

In [ ]:
# ============================================================
# HMM state distribution by V11/HDBSCAN cluster
# ============================================================

state_by_v11_cluster = (
    hmm_df
    .groupby(["stratum", "cluster_key", "hmm_state"])
    .size()
    .reset_index(name="n_rows")
)

state_by_v11_cluster["share_within_cluster"] = (
    state_by_v11_cluster["n_rows"]
    / state_by_v11_cluster.groupby(["stratum", "cluster_key"])["n_rows"].transform("sum")
)

display(
    state_by_v11_cluster
    .sort_values(["stratum", "cluster_key", "share_within_cluster"], ascending=[True, True, False])
    .head(100)
)

## Optional risk-oriented analysis

Because we do not automatically know which HMM state is “risky,” this section lets you manually define risky states after inspecting the emission probabilities and state profiles.

Example:
- If `hmm_state_0` mostly emits a low/no-activity GMM cluster, it may be a risky or inactive state.
- If `hmm_state_2` mostly emits high-intent weekly behavior, it may be a healthy state.

Update `RISKY_HMM_STATES` after interpretation.

In [ ]:
# ============================================================
# Manual risky state assignment
# ============================================================

# Update after inspecting emission_probs and state_profiles.
RISKY_HMM_STATES = []  # example: [0]

hmm_df["is_risky_hmm_state"] = hmm_df["hmm_state"].isin(RISKY_HMM_STATES).astype(int)

if RISKY_HMM_STATES:
    risk_by_cluster = (
        hmm_df
        .groupby(["stratum", "cluster_key"])
        .agg(
            n_weekly_rows=("developer_id", "size"),
            n_developers=("developer_id", "nunique"),
            risk_state_share=("is_risky_hmm_state", "mean")
        )
        .reset_index()
        .sort_values("risk_state_share", ascending=False)
    )

    display(risk_by_cluster.head(50))
else:
    print("RISKY_HMM_STATES is empty. Inspect emission_probs/state_profiles first, then fill this in.")

In [ ]:
# ============================================================
# Save small experiment outputs locally
# We are NOT saving model assignments to DuckDB yet because this is exploratory.
# ============================================================

results_df.to_csv(OUTPUT_DIR / "hmm_categorical_model_comparison.csv", index=False)
state_profiles.to_csv(OUTPUT_DIR / "hmm_categorical_state_profiles.csv", index=False)
transition_matrix.to_csv(OUTPUT_DIR / "hmm_categorical_transition_matrix.csv")
transition_long.to_csv(OUTPUT_DIR / "hmm_categorical_transition_matrix_long.csv", index=False)
emission_probs.to_csv(OUTPUT_DIR / "hmm_categorical_emission_probabilities.csv")
state_by_stratum.to_csv(OUTPUT_DIR / "hmm_categorical_state_by_stratum.csv", index=False)
state_by_v11_cluster.to_csv(OUTPUT_DIR / "hmm_categorical_state_by_v11_cluster.csv", index=False)

print("Saved summary outputs to:", OUTPUT_DIR)

# Interpretation notes to fill in after running

After running this notebook, answer:

1. What does each weekly GMM cluster mean behaviorally?
2. What does each HMM hidden state emit most often?
3. Which HMM states look healthy, transitional, or risky?
4. Which V11/HDBSCAN clusters spend the most time in risky HMM states?
5. Which clusters show the strongest opportunity for personalized outreach?

Suggested final business framing:

```text
The weekly GMM model gives each developer-week an observed behavior state. The categorical HMM uses these observed weekly states to learn hidden journey states and transition probabilities over time. By joining the HMM states back to the V11 lifecycle/HDBSCAN clusters, we can identify which developer segments are following healthy adoption paths versus which segments are trending toward lower engagement or at-risk behavior.
```